### imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import talib
import yfinance as yf
import os

sns.set_theme(style="darkgrid")

### Load from your CSV files

In [ ]:
# If you have local CSV files in data/raw/
import os

stock_files = os.listdir("../data/raw/")
print(stock_files)  # check what files you have

# Load one stock as example
df = pd.read_csv("../data/raw/AAPL_historical.csv", parse_dates=["Date"])
df.sort_values("Date", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

### Data Cleaning

In [ ]:
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

# Ensure correct types
for col in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows with missing OHLCV
df.dropna(subset=["Open", "High", "Low", "Close", "Volume"], inplace=True)

# Sort by date
df.sort_values("Date", inplace=True)
df.reset_index(drop=True, inplace=True)

print("\n✅ After cleaning:", df.shape)

### Moving Averages (SMA & EMA)

In [ ]:
# Simple Moving Averages
df["SMA_20"]  = talib.SMA(df["Close"], timeperiod=20)
df["SMA_50"]  = talib.SMA(df["Close"], timeperiod=50)
df["SMA_200"] = talib.SMA(df["Close"], timeperiod=200)

# Exponential Moving Averages
df["EMA_12"] = talib.EMA(df["Close"], timeperiod=12)
df["EMA_26"] = talib.EMA(df["Close"], timeperiod=26)

print("✅ Moving averages computed")
df[["Date", "Close", "SMA_20", "SMA_50", "EMA_12"]].tail(10)

### RSI (Relative Strength Index)

In [ ]:
df["RSI_14"] = talib.RSI(df["Close"], timeperiod=14)

print("RSI Stats:")
print(df["RSI_14"].describe())

### MACD

In [ ]:
# MACD line, Signal line, Histogram
df["MACD"], df["MACD_signal"], df["MACD_hist"] = talib.MACD(
    df["Close"],
    fastperiod=12,
    slowperiod=26,
    signalperiod=9
)

print("✅ MACD computed")
df[["Date", "MACD", "MACD_signal", "MACD_hist"]].tail(10)

###  Bollinger Bands (Bonus)

In [ ]:
df["BB_upper"], df["BB_middle"], df["BB_lower"] = talib.BBANDS(
    df["Close"],
    timeperiod=20,
    nbdevup=2,
    nbdevdn=2
)

print("✅ Bollinger Bands computed")

###  Price + Moving Averages

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df["Date"], df["Close"],   label="Close Price", color="black",    linewidth=1.2)
ax.plot(df["Date"], df["SMA_20"],  label="SMA 20",      color="blue",     linewidth=1)
ax.plot(df["Date"], df["SMA_50"],  label="SMA 50",      color="orange",   linewidth=1)
ax.plot(df["Date"], df["SMA_200"], label="SMA 200",     color="red",      linewidth=1)
ax.plot(df["Date"], df["EMA_12"],  label="EMA 12",      color="green",    linewidth=1, linestyle="--")

ax.set_title("AAPL — Close Price with Moving Averages", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()

### RSI Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Price
ax1.plot(df["Date"], df["Close"], color="black", linewidth=1)
ax1.set_title("AAPL — Close Price", fontsize=13)
ax1.set_ylabel("Price (USD)")

# RSI
ax2.plot(df["Date"], df["RSI_14"], color="purple", linewidth=1)
ax2.axhline(70, color="red",   linestyle="--", linewidth=1, label="Overbought (70)")
ax2.axhline(30, color="green", linestyle="--", linewidth=1, label="Oversold (30)")
ax2.fill_between(df["Date"], df["RSI_14"], 70,
                 where=(df["RSI_14"] >= 70), alpha=0.3, color="red")
ax2.fill_between(df["Date"], df["RSI_14"], 30,
                 where=(df["RSI_14"] <= 30), alpha=0.3, color="green")
ax2.set_title("RSI (14)", fontsize=13)
ax2.set_ylabel("RSI")
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.show()

### MACD Plot

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Price
ax1.plot(df["Date"], df["Close"], color="black", linewidth=1)
ax1.set_title("AAPL — Close Price", fontsize=13)
ax1.set_ylabel("Price (USD)")

# MACD
ax2.plot(df["Date"], df["MACD"],        label="MACD",   color="blue",  linewidth=1)
ax2.plot(df["Date"], df["MACD_signal"], label="Signal", color="orange",linewidth=1)

# Color histogram bars
colors = ["green" if v >= 0 else "red" for v in df["MACD_hist"].fillna(0)]
ax2.bar(df["Date"], df["MACD_hist"], color=colors, alpha=0.5, label="Histogram")

ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_title("MACD (12, 26, 9)", fontsize=13)
ax2.set_ylabel("MACD Value")
ax2.legend()

plt.tight_layout()
plt.show()

### Bollinger Bands

In [ ]:
# Plot last 6 months for clarity
recent = df.tail(180)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(recent["Date"], recent["Close"],     label="Close",        color="black", linewidth=1.2)
ax.plot(recent["Date"], recent["BB_upper"],  label="Upper Band",   color="red",   linewidth=1, linestyle="--")
ax.plot(recent["Date"], recent["BB_middle"], label="Middle (SMA)", color="blue",  linewidth=1)
ax.plot(recent["Date"], recent["BB_lower"],  label="Lower Band",   color="green", linewidth=1, linestyle="--")

ax.fill_between(recent["Date"], recent["BB_upper"], recent["BB_lower"],
                alpha=0.1, color="blue")

ax.set_title("AAPL — Bollinger Bands (Last 6 Months)", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.tight_layout()
plt.show()

### Compute indicators for all tickers

In [ ]:
def compute_indicators(df):
    df = df.copy()
    df.sort_values("Date", inplace=True)
    
    close = df["Close"]
    
    df["SMA_20"]       = talib.SMA(close, timeperiod=20)
    df["SMA_50"]       = talib.SMA(close, timeperiod=50)
    df["EMA_12"]       = talib.EMA(close, timeperiod=12)
    df["EMA_26"]       = talib.EMA(close, timeperiod=26)
    df["RSI_14"]       = talib.RSI(close, timeperiod=14)
    df["MACD"], df["MACD_signal"], df["MACD_hist"] = talib.MACD(close, 12, 26, 9)
    df["BB_upper"], df["BB_middle"], df["BB_lower"] = talib.BBANDS(close, 20)
    
    # Daily return using Adj Close
    df["Daily_Return"] = df["Adj Close"].pct_change() * 100
    
    return df

# Apply to all stocks
for ticker in tickers:
    stock_data[ticker] = compute_indicators(stock_data[ticker])
    print(f"✅ Indicators computed for {ticker}")